In [1]:
# -*- coding: utf-8 -*-
"""
因子: fin_abcdfgh_ck_v3 池簇102 (尾盘收益上半场取Hurst否则取午后收益5日衰减
对数压缩作门控源, 上半场取偏度10日波动, 否则取尾盘收益)

来源: gp_survivors_fin_abcdfgh_ck_v3.json 簇102 (h批 seed72 产物, 1034池315簇
全量开箱 2026-07-30), holdout(2024H2) ic +0.0346 / ir +0.734 / t +8.18,
压力ic +0.0335, LS夏普(全) +11.19, dir=-1 (提交口径取负翻正)。

表达式 (Z[·] = 逐日截面 winsorize 1%/99% + zscore):
    f = -cond( cs_top_q( cond( cs_top_q(Z[ret_tail30], 0.5), Z[hurst_ret],
                               log_p(ts_decay_5(Z[ret_pm])) ), 0.5 ),
               ts_std_10(Z[ret_skew]), Z[ret_tail30] )
    cs_top_q(x,0.5) = 截面 rank(x)>0.5 的 {0,1} 指示 (rank NaN 处 NaN)
    cond(b,x,y) = b==1 -> x; b==0 -> y; b NaN -> NaN
    ts_decay_5 = 线性衰减加权均值 (权重5..1, 任一缺失->NaN)
    ts_std_10 = 10日滚动样本标准差 (满窗口); log_p(x)=sign(x)·ln(1+|x|)

终端口径 (240根完整分钟日; r_t=ln(close_t/close_{t-1}), 239个):
    hurst_ret = 对数收盘价路径聚合方差法 Hurst (lag 2..19, STDDEV_POP,
                slope(ln tau ~ ln lag); 任一 tau<=0 -> NaN)
    ret_pm = ln(close_15:00/close_11:30); ret_skew = 日内分钟收益样本偏度
    ret_tail30 = ln(close_15:00/close_14:30)
ts 最大回看 10 交易日, 查询起点前置 40 自然日, 输出裁剪回评测区间。
本地验证: 60天 spearman min=median=1.000000 (scripts/verify_ck_factors.py 102)

确定性加固 (2026-07-31, 平台截断式未来函数检测误报"疑似有未来函数"复盘):
本因子无未来依赖 (逐日聚合 + 10日后向滚动), 误报源自两跑数值不可复现;
且旧版两层窗口 CTE (m 层 LAG(close) + d 层 18 个 LAG(lc,l)) 需两次全表
排序, 有平台资源超限风险。修法: 1) 压成单 pass —— d{l} 改
LN(close) - LN(LAG(close,l)) 与 lc - LAG(lc,l) 逐位等价 (同样的两个 LN
值相减, NULL 传播一致), 所有 LAG 共用一个 WINDOW spec; 2) 18 个
STDDEV_POP 与 SKEWNESS 改 DECIMAL 矩量和 (并行聚合浮点合并顺序免疫),
统计量 pandas 重算 (公式与原生聚合相对差 <3e-15); 3) LAST 改
MAX(CASE 15:00) (240 格日逐位等价, 与扫描/合并顺序无关); 4) 终端 f32
量化 + 输出 round(8) + 行序固定 (截断检测容差保险)。

零价保护 (2026-08-05, 平台 `OutOfRangeException: cannot take logarithm of
zero` 复盘): 平台真表在停牌/未上市分钟给 close=0, DuckDB 的 LN(0) **抛异常
而非返回 NULL**, 于是 `LN(close)` / `LN(close/LAG(close))` 整条查询炸掉。
本地 dai 兼容层 (`dai.py::_stock_view_sql`) 把非正价映射成 NULL 且本地分钟
缓存零价行数为 0, 故金标准脚本测不到。修法 = 入口 CTE 把非正价映射成
NULL (与本地口径一致), LN 只吃 NULL 不吃 0 —— 对"旧版能跑通的输入"是恒等
变换 (实测 2024-03 全月 21000 格 maxdiff=0), 回归验证见
scripts/verify_zero_close_guard.py。

覆盖率保护 (2026-08-05): 当日池内覆盖率 (当日非空因子值数 / 当日宇宙股票数)
不足 65% 时, 当日缺失的股票沿用**前一交易日**的因子值 —— 防 2020-02-03 千股
跌停一类整截面失效触发平台 run error (判据 = 最差单日覆盖深度)。只读过去,
无前视; 逐日推进故连续低覆盖日顺次传递; 覆盖率达标日的个股缺失不回填 (那是
个股停牌, 不是截面失效)。若区间内每日覆盖率都达标, 本段是恒等变换。
"""

_COV_MIN = 0.65   # 当日池内覆盖率下限, 低于此值的交易日启用前一日回填


def main(datasources, start_date, end_date):
    import numpy as np
    import pandas as pd
    import dai

    bar1m_table = datasources["bar1m"]

    univ = dai.query(
        "SELECT DISTINCT instrument FROM bigalpha_2026_instruments",
        filters={"date": ["2019-01-01", "2099-12-31"]},
    ).df()["instrument"]

    # LN(close)-LN(LAG(close,l)) 与 lc-LAG(lc,l) 逐位等价 (同样的两个 LN 值
    # 相减, NULL 传播一致), 免第二层窗口 CTE
    lag_cols = ",\n               ".join(
        f"LN(close) - LN(LAG(close, {l}) OVER w) AS d{l}" for l in range(2, 20))
    # DECIMAL 矩量和: 与并行聚合的浮点合并顺序无关, 两跑逐位一致
    mom_cols = ",\n           ".join(
        f"COUNT(d{l}) AS n{l},\n           "
        f"CAST(SUM(CAST(d{l} AS DECIMAL(38,25))) AS DOUBLE) AS s{l},\n           "
        f"CAST(SUM(CAST(d{l} * d{l} AS DECIMAL(38,25))) AS DOUBLE) AS q{l}"
        for l in range(2, 20))
    bar_sql = f"""
    WITH px AS (
        -- 零价保护: 平台真表在停牌/未上市分钟给 close=0, 而 DuckDB 的 LN(0)
        -- 抛 OutOfRangeException (不是返回 NULL); 本地 dai 兼容层把非正价
        -- 映射成 NULL 所以本地复现不了。这里统一映射成 NULL —— 对任何
        -- "旧版能跑通的输入" 是恒等变换, 且与本地/挖掘端口径逐位一致
        SELECT date, instrument, CASE WHEN close > 0 THEN close END AS close
        FROM {bar1m_table}
    ),
    m AS (
        SELECT date::DATE AS day, instrument, date::TIME AS tm, date, close,
               LN(close / LAG(close) OVER w) AS r,
               {lag_cols}
        FROM px
        WINDOW w AS (PARTITION BY date::DATE, instrument ORDER BY date)
    )
    SELECT day::DATETIME AS date, instrument,
           COUNT(r) AS sk_n,
           CAST(SUM(CAST(r AS DECIMAL(38,25))) AS DOUBLE) AS sk_s1,
           CAST(SUM(CAST(r * r AS DECIMAL(38,25))) AS DOUBLE) AS sk_s2,
           CAST(SUM(CAST(r * r * r AS DECIMAL(38,25))) AS DOUBLE) AS sk_s3,
           -- MAX(CASE 15:00) 与 LAST(ORDER BY date) 在 240 格日逐位等价,
           -- 与扫描/合并顺序无关
           LN(MAX(CASE WHEN tm = TIME '15:00:00' THEN close END)
              / MAX(CASE WHEN tm = TIME '11:30:00' THEN close END)) AS ret_pm,
           LN(MAX(CASE WHEN tm = TIME '15:00:00' THEN close END)
              / MAX(CASE WHEN tm = TIME '14:30:00' THEN close END)) AS ret_tail30,
           {mom_cols}
    FROM m
    GROUP BY day, instrument
    HAVING COUNT(*) = 240
    """
    buf_start = (pd.Timestamp(start_date) - pd.Timedelta(days=40)).strftime("%Y-%m-%d")
    parts = []
    for q in pd.period_range(start=buf_start, end=end_date, freq="Q"):
        qs = max(q.start_time, pd.Timestamp(buf_start))
        qe = min(q.end_time.normalize(), pd.Timestamp(end_date))
        filt = {"date": [qs.strftime("%Y-%m-%d 00:00:00"),
                         qe.strftime("%Y-%m-%d 23:59:59")]}
        parts.append(dai.query(bar_sql, filters=filt).df())
    px = pd.concat(parts, ignore_index=True)
    px["date"] = px["date"].astype("datetime64[ns]")

    lags = np.arange(2, 20)
    with np.errstate(all="ignore"):
        # 矩量和重算 STDDEV_POP (n=0 -> NaN, 负方差截 0)
        for l in lags:
            n = px[f"n{l}"].to_numpy(np.float64)
            s = px[f"s{l}"].to_numpy(np.float64)
            qq = px[f"q{l}"].to_numpy(np.float64)
            nn = np.maximum(n, 1.0)
            px[f"t{l}"] = np.where(
                n >= 1, np.sqrt(np.maximum(qq / nn - (s / nn) ** 2, 0.0)),
                np.nan)
        # 矩量和重算 SKEWNESS (样本偏度, n<3 或 m2<=0 -> NaN, 与原生一致)
        n = px["sk_n"].to_numpy(np.float64)
        s1 = px["sk_s1"].to_numpy(np.float64)
        s2 = px["sk_s2"].to_numpy(np.float64)
        s3 = px["sk_s3"].to_numpy(np.float64)
        m2 = s2 / n - (s1 / n) ** 2
        m3 = (s3 - 3.0 * s2 * s1 / n + 2.0 * s1 ** 3 / n ** 2) / n
        px["ret_skew"] = np.where(
            (n >= 3) & (m2 > 0),
            np.sqrt(n * (n - 1.0)) / (n - 2.0) * m3 / m2 ** 1.5, np.nan)

    # hurst 回归逻辑不动: slope(ln tau ~ ln lag)
    lxc = np.log(lags) - np.log(lags).mean()
    taus = px[[f"t{l}" for l in lags]].to_numpy(dtype=np.float64)
    with np.errstate(all="ignore"):
        ly = np.log(taus)
        hurst = ly @ lxc / (lxc ** 2).sum()
    hurst[~(np.isfinite(ly).all(axis=1) & (taus > 0).all(axis=1))] = np.nan
    px["hurst_ret"] = hurst
    px = px.drop(columns=[f"t{l}" for l in lags]
                 + [f"{p}{l}" for l in lags for p in ("n", "s", "q")]
                 + ["sk_n", "sk_s1", "sk_s2", "sk_s3"])

    terms = ["hurst_ret", "ret_pm", "ret_skew", "ret_tail30"]
    for c in terms:   # 终端 f32 量化: 吞掉平台侧末位抖动 (硬分支前置消抖)
        px[c] = px[c].astype(np.float32).astype(np.float64)
    didx = pd.DatetimeIndex(np.sort(px["date"].unique()))
    codes = np.sort(univ.unique())
    piv = {c: px.pivot_table(index="date", columns="instrument", values=c)
             .reindex(index=didx, columns=codes)
           for c in terms}

    def zscore_rows(df):
        m = df.to_numpy(dtype=np.float64)
        with np.errstate(all="ignore"):
            lo = np.nanquantile(m, 0.01, axis=1, keepdims=True)
            hi = np.nanquantile(m, 0.99, axis=1, keepdims=True)
            m = np.clip(m, lo, hi)
            mu = np.nanmean(m, axis=1, keepdims=True)
            sd = np.nanstd(m, axis=1, keepdims=True)
            m = np.where(sd > 0, (m - mu) / sd, 0.0)
        return pd.DataFrame(m, index=df.index, columns=df.columns)

    z = {c: zscore_rows(piv[c]) for c in terms}

    def csr(x):
        return pd.DataFrame(x).rank(axis=1, pct=True).to_numpy()

    with np.errstate(all="ignore"):
        dec = (sum((5 - i) * z["ret_pm"].shift(i) for i in range(5)) / 15.0) \
            .to_numpy()
        lp = np.sign(dec) * np.log1p(np.abs(dec))
        r1 = csr(z["ret_tail30"].to_numpy())
        inner = np.where(r1 > 0.5, z["hurst_ret"].to_numpy(), lp)
        inner[np.isnan(r1)] = np.nan
        r2 = csr(inner)
        x = z["ret_skew"].rolling(10, min_periods=10).std().to_numpy()
        f = np.where(r2 > 0.5, x, z["ret_tail30"].to_numpy())
        f[np.isnan(r2)] = np.nan
    f = np.where(np.isfinite(f), f, np.nan)
    f = -f          # dir=-1: 提交口径统一为正 IC

    out = pd.DataFrame(f, index=didx, columns=codes) \
        .stack().rename("factor").reset_index()
    out.columns = ["date", "instrument", "factor"]
    out = out[(out["date"] >= start_date) & (out["date"] <= end_date)]
    out["factor"] = out["factor"].round(8)   # 吸收浮点末位抖动 (截断检测容差保险)

    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    out = pool.merge(out, on=["date", "instrument"], how="left") \
        .sort_values(["date", "instrument"], kind="mergesort") \
        .reset_index(drop=True)              # 输出行序确定 (pool 的 SQL 行序不定)

    # ---- 覆盖率保护 (仅此一段, 上面的计算逻辑一律不动)
    # 当日池内覆盖率 = 当日非空因子值数 / 当日宇宙股票数; 不足 _COV_MIN 的
    # 交易日, 其缺失格子沿用前一交易日的值 (只读过去, 无前视)。逐日推进,
    # 连续低覆盖日顺次传递; 覆盖率达标日的缺失不回填
    uniq = out.drop_duplicates(["date", "instrument"])
    w = uniq.pivot(index="date", columns="instrument", values="factor")
    if len(w) > 1:
        n_pool = uniq.groupby("date").size().reindex(w.index).to_numpy(np.float64)
        cov = w.notna().sum(axis=1).to_numpy(np.float64) / n_pool
        vals = w.to_numpy(dtype=np.float64, copy=True)
        for i in range(1, len(vals)):
            if cov[i] < _COV_MIN:
                miss = np.isnan(vals[i])
                vals[i][miss] = vals[i - 1][miss]
        fill = pd.DataFrame(vals, index=w.index, columns=w.columns) \
            .stack().rename("factor").reset_index()
        fill.columns = ["date", "instrument", "factor"]
        # merge how="left" 保持左表行序, 上面固定好的行序不变
        out = out.drop(columns="factor") \
            .merge(fill, on=["date", "instrument"], how="left")
    return out[["date", "instrument", "factor"]]


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m',
        'financial': 'bigalpha_2026_financial'
        }
    start_date = '2020-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )


[2026-08-05 15:05:45] [info     ] 计算因子，区间：2020-01-01 00:00:00 ~ 2024-12-31 23:59:59
[2026-08-05 15:20:07] [info     ] 读取因子库，区间：2020-01-01 00:00:00 ~ 2024-12-31 23:59:59
[2026-08-05 15:20:08] [warning  ] bigalpha_eval._latest version='v4' (use ._latest for dev only, not for prod)


In [10]:
    import numpy as np
    import pandas as pd

    # 保留需要的字段
    df = factor_data[["date", "instrument", "factor"]].copy()

    # 日期统一为 YYYY-MM-DD
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()

    # 转成数值；NaN、inf、-inf 都视为无效因子
    df["factor"] = pd.to_numeric(df["factor"], errors="coerce")
    df["is_valid"] = np.isfinite(df["factor"])

    # 分日统计
    daily_coverage = (
        df.groupby("date", as_index=False)
        .agg(
            total_count=("instrument", "nunique"),
            valid_count=("is_valid", "sum"),
        )
        .sort_values("date")
        .reset_index(drop=True)
    )

    daily_coverage["missing_count"] = (
        daily_coverage["total_count"] - daily_coverage["valid_count"]
    )

    daily_coverage["coverage_rate"] = (
        daily_coverage["valid_count"] / daily_coverage["total_count"]
    )

    # 逐日打印
    print("=" * 82)
    print(f"{'日期':<12} {'股票总数':>10} {'有效数量':>10} {'缺失数量':>10} {'覆盖率':>10}  状态")
    print("=" * 82)

    for row in daily_coverage.itertuples(index=False):
        coverage_pct = row.coverage_rate * 100

        if row.coverage_rate < 0.80:
            status = "⚠️ 低于80%"
        else:
            status = "正常"

        print(
            f"{row.date.strftime('%Y-%m-%d'):<12} "
            f"{row.total_count:>10d} "
            f"{row.valid_count:>10d} "
            f"{row.missing_count:>10d} "
            f"{coverage_pct:>9.2f}%  "
            f"{status}"
        )

    print("=" * 82)

    # 汇总异常日期
    low_coverage = daily_coverage[daily_coverage["coverage_rate"] < 0.80]

    print()
    print(f"交易日总数：{len(daily_coverage)}")
    print(f"覆盖率低于 80% 的交易日数：{len(low_coverage)}")
    print(f"最低覆盖率：{daily_coverage['coverage_rate'].min() * 100:.2f}%")
    print(f"平均覆盖率：{daily_coverage['coverage_rate'].mean() * 100:.2f}%")

    if not low_coverage.empty:
        print("\n⚠️ 覆盖率低于 80% 的日期：")
        for row in low_coverage.itertuples(index=False):
            print(
                f"  {row.date.strftime('%Y-%m-%d')}："
                f"{row.coverage_rate * 100:.2f}% "
                f"（有效 {row.valid_count}/{row.total_count}，"
                f"缺失 {row.missing_count}）"
            )
    else:
        print("\n所有交易日的因子覆盖率均不低于 80%。")

日期                 股票总数       有效数量       缺失数量        覆盖率  状态
2020-03-02         1000        985         15     98.50%  正常
2020-03-03         1000        983         17     98.30%  正常
2020-03-04         1000        986         14     98.60%  正常
2020-03-05         1000        980         20     98.00%  正常
2020-03-06         1000        987         13     98.70%  正常
2020-03-09         1000        988         12     98.80%  正常
2020-03-10         1000        988         12     98.80%  正常
2020-03-11         1000        990         10     99.00%  正常
2020-03-12         1000        986         14     98.60%  正常
2020-03-13         1000        992          8     99.20%  正常
2020-03-16         1000        993          7     99.30%  正常
2020-03-17         1000        989         11     98.90%  正常
2020-03-18         1000        987         13     98.70%  正常
2020-03-19         1000        989         11     98.90%  正常
2020-03-20         1000        990         10     99.00%  正常
2020-03-23         1000 